In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from dataclasses import dataclass
from typing import Optional, Dict, Any

from sklearn.metrics import mean_absolute_error

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModel,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed,
)

In [2]:
MODEL_NAME = "answerdotai/ModernBERT-large"
TRAIN_CSV = "/content/ielts_train_df.csv"
VAL_CSV   = "/content/ielts_val_df.csv"
TEST_CSV  = "/content/ielts_test_df.csv"

MAX_LENGTH = 512
SEED = 42
BATCH_SIZE = 8
GRAD_ACCUM = 4
LR = 1e-6
EPOCHS = 10
WEIGHT_DECAY = 0.01
OUTPUT_DIR = "./deberta_ielts_4criteria"

TARGET_COLS = ["TR", "CC", "LR", "GRA"]

set_seed(SEED)

In [3]:
# --- CẬP NHẬT CELL 3: LÀM SẠCH DỮ LIỆU TRIỆT ĐỂ ---

# 1. Load dữ liệu gốc
train_df = pd.read_csv(TRAIN_CSV, engine="python", on_bad_lines="skip")
val_df = pd.read_csv(VAL_CSV, engine="python", on_bad_lines="skip") if os.path.exists(VAL_CSV) else None
test_df = pd.read_csv(TEST_CSV, engine="python", on_bad_lines="skip") if os.path.exists(TEST_CSV) else None

# 2. Tách tập validation nếu chưa có
if val_df is None:
    from sklearn.model_selection import train_test_split
    train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=SEED)

needed_cols = ["prompt", "essay"] + TARGET_COLS

def robust_clean_df(df):
    if df is None: return None
    # Chỉ lấy các cột cần thiết
    df = df[needed_cols].copy()

    # Ép kiểu nhãn sang numeric và loại bỏ NaN ở nhãn
    for col in TARGET_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=TARGET_COLS).reset_index(drop=True)

    # QUAN TRỌNG: Ép kiểu essay sang string và loại bỏ khoảng trắng thừa
    df['essay'] = df['essay'].astype(str).str.strip()

    # Loại bỏ các bài viết quá ngắn (dưới 20 ký tự thường là dữ liệu lỗi)
    df = df[df['essay'].str.len() > 20].reset_index(drop=True)

    # Giới hạn điểm số trong dải hợp lệ [0.0, 9.0] để tránh outlier làm nổ gradient
    for col in TARGET_COLS:
        df[col] = df[col].clip(0.0, 9.0)

    return df

# Áp dụng hàm làm sạch cho các tập dữ liệu
train_df = robust_clean_df(train_df)
val_df = robust_clean_df(val_df)
test_df = robust_clean_df(test_df)

print(f"Train shape: {train_df.shape if train_df is not None else 0}")
print(f"Val shape: {val_df.shape if val_df is not None else 0}")
print(train_df[TARGET_COLS].head())

Train shape: (6618, 6)
Val shape: (827, 6)
    TR   CC   LR  GRA
0  6.0  6.0  6.0  6.0
1  6.5  6.5  6.5  6.5
2  7.0  7.0  7.0  7.0
3  5.0  5.0  5.0  5.0
4  4.5  4.5  4.5  4.5


In [4]:
def build_input_text(row):
    prompt = str(row["prompt"]).strip()
    essay = str(row["essay"]).strip()
    return f"[PROMPT]\n{prompt}\n\n[ESSAY]\n{essay}"

train_df["text"] = train_df.apply(build_input_text, axis=1)
val_df["text"] = val_df.apply(build_input_text, axis=1)

if test_df is not None:
    test_df["text"] = test_df.apply(build_input_text, axis=1)

def make_labels(df):
    df = df.copy()
    df["labels"] = (df[TARGET_COLS] / 9.0).values.tolist()
    return df

train_df = make_labels(train_df)
val_df = make_labels(val_df)
if test_df is not None:
    test_df = make_labels(test_df)

In [5]:
train_ds = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[["text", "labels"]], preserve_index=False)

dataset_dict = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
})

if test_df is not None:
    test_ds = Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False)
    dataset_dict["test"] = test_ds

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_ds = dataset_dict.map(tokenize_fn, batched=True)
# QUAN TRỌNG: Chỉ giữ lại các cột số (input_ids, attention_mask, labels)
tokenized_ds = tokenized_ds.remove_columns(["text"])
tokenized_ds.set_format(type="torch")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Map:   0%|          | 0/6618 [00:00<?, ? examples/s]

Map:   0%|          | 0/827 [00:00<?, ? examples/s]

Map:   0%|          | 0/828 [00:00<?, ? examples/s]

In [7]:
class MeanPooling(nn.Module):
    def __init__(self):
        super(MeanPooling, self).__init__()
    def forward(self, last_hidden_state, attention_mask):
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        return sum_embeddings / sum_mask

class DebertaForMultiRegression(nn.Module):
    def __init__(self, model_name: str, num_labels: int = 4, dropout: float = 0.2):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config)
        self.pooler = MeanPooling() # Dùng lớp pooling chuẩn

        hidden_size = self.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, num_labels)

        # Khởi tạo head với bias=6.0 để giữ ổn định lúc đầu
        nn.init.normal_(self.head.weight, std=0.01)
        nn.init.constant_(self.head.bias, 0.45)

        self.loss_fn = nn.SmoothL1Loss() # Source chuyên nghiệp thường dùng cái này thay vì Huber

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = self.pooler(outputs.last_hidden_state, attention_mask)
        logits = self.head(self.dropout(pooled_output))
        if torch.isnan(outputs.last_hidden_state).any() or torch.isinf(outputs.last_hidden_state).any():
            raise ValueError("NaN/Inf detected in backbone output")
        if torch.isnan(logits).any() or torch.isinf(logits).any():
            print("Logits min/max:", logits.min().item(), logits.max().item())
            raise ValueError("NaN/Inf detected in logits")

        loss = None
        if labels is not None:
            #print("Labels min/max:", labels.min().item(), labels.max().item())
            loss = self.loss_fn(logits, labels.float())

            if torch.isnan(loss).any() or torch.isinf(loss).any():
                #print("Logits min/max:", logits.min().item(), logits.max().item())
                #print("Labels min/max:", labels.min().item(), labels.max().item())
                raise ValueError("NaN/Inf detected in loss")

        return {"loss": loss, "logits": logits}

model = DebertaForMultiRegression(MODEL_NAME)

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
from dataclasses import dataclass
from typing import Any, Dict, List
import torch

@dataclass
class RegressionCollator:
    tokenizer: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # nếu labels đã là tensor thì stack
        if isinstance(features[0]["labels"], torch.Tensor):
            labels = torch.stack([f["labels"] for f in features]).float()
        else:
            labels = torch.tensor([f["labels"] for f in features], dtype=torch.float)

        batch = self.tokenizer.pad(
            [{k: v for k, v in f.items() if k != "labels"} for f in features],
            padding=True,
            return_tensors="pt"
        )
        batch["labels"] = labels
        return batch

data_collator = RegressionCollator(tokenizer)

In [9]:
from sklearn.metrics import cohen_kappa_score

def round_to_half(x):
    return np.round(x * 2) / 2


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]

    if np.isnan(preds).any():
        print("WARNING: preds contains NaN")

    preds = preds * 9.0
    labels = labels * 9.0

    preds = np.clip(preds, 0.0, 9.0)
    labels = np.clip(labels, 0.0, 9.0)

    preds_half = round_to_half(preds)

    maes = [mean_absolute_error(labels[:, i], preds_half[:, i]) for i in range(4)]

    qwks = []
    for i in range(4):
        y_true = (labels[:, i] * 2).astype(int)
        y_pred = (preds_half[:, i] * 2).astype(int)
        score = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        qwks.append(score)

    return {
        "mean_mae": np.mean(maes),
        "mean_qwk": np.mean(qwks),
        "tr_qwk": qwks[0],
        "cc_qwk": qwks[1],
        "lr_qwk": qwks[2],
        "gra_qwk": qwks[3],
        "within_0.5_acc": np.mean(np.abs(preds_half - labels) <= 0.5),
    }

In [10]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0,
    load_best_model_at_end=True,
    metric_for_best_model="mean_qwk",
    greater_is_better=True,
    fp16=False,
    bf16=True,
    report_to="none",
    save_total_limit=2,
    max_grad_norm=0.8,           # CHẶN GRADIENT TẠI ĐÂY
    gradient_checkpointing=False,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [11]:
# --- SỬA CELL 10 ---
from transformers import get_cosine_schedule_with_warmup

def get_optimizer_params(model, encoder_lr, decoder_lr, weight_decay=0.0):
    no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]
    optimizer_parameters = [
        {'params': [p for n, p in model.backbone.named_parameters() if not any(nd in n for nd in no_decay)],
         'lr': encoder_lr, 'weight_decay': weight_decay},
        {'params': [p for n, p in model.backbone.named_parameters() if any(nd in n for nd in no_decay)],
         'lr': encoder_lr, 'weight_decay': 0.0},
        {'params': [p for n, p in model.named_parameters() if "head" in n and not any(nd in n for nd in no_decay)],
         'lr': decoder_lr, 'weight_decay': weight_decay},
        {'params': [p for n, p in model.named_parameters() if "head" in n and any(nd in n for nd in no_decay)],
         'lr': decoder_lr, 'weight_decay': 0.0},
    ]
    return optimizer_parameters

# Tính toán số bước (Step)
num_training_steps = (len(tokenized_ds["train"]) // BATCH_SIZE) * EPOCHS

# Cấu hình LR tối ưu: 2e-5 cho backbone và 5e-5 cho head
optimizer_parameters = get_optimizer_params(
    model,
    encoder_lr=1e-5,
    decoder_lr=5e-5,
    weight_decay=0.01
)
optimizer = torch.optim.AdamW(optimizer_parameters)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(num_training_steps * 0.1),
    num_training_steps=num_training_steps
)

In [12]:
# --- SỬA CELL 11 ---
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    # Ép kiểu tuple rõ ràng để tránh lỗi "cannot unpack"
    optimizers=(optimizer, scheduler),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)

In [13]:
from torch.utils.data import DataLoader

debug_loader = DataLoader(
    tokenized_ds["train"],
    batch_size=2,
    collate_fn=data_collator
)

batch = next(iter(debug_loader))
print(batch.keys())
print(batch["labels"])
print(batch["labels"].shape)
print(batch["labels"].dtype)

KeysView({'input_ids': tensor([[50281,    60, 35672,  5736,    62,   187, 10195,  4343,   403,  9100,
           247,  5699,  2408,   273,  2583,   327,  8109,   616, 21607,   281,
          1379,   629,   275,   690, 11762,  9001, 31067,    15, 19810,  9059,
           326,   352,   651,   320,  1805,   604,   841,  4343,   476,  6947,
           253,  2583,   327,  2151,   281,  1379,   629,   275,  9001,    15,
          1916,   752,  6070,   513,   368,  5194,   390, 14936,    32,   187,
           187,    60,  1410,  4576,    58,    62,   187,  2512,   403,  2257,
           273,  4343,   275,   253,  1533,   665,   403,   970,  1270,  2408,
           273,  2583,   327, 32933,   273,   616,  6671,    15,   309,  5194,
           326,   597,   943,  1265,  9100,  2583,   327,  2151,  9678,  4712,
           275,   326,  1039,   597,   588,  1265,   253, 31868,   273,  2393,
          8661,   275,   616,  1495,  4404,  3958,    15,   281,  1265,   342,
          1429,   373,  9001,

In [14]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.
W0307 18:00:23.434000 10416 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


Epoch,Training Loss,Validation Loss,Mean Mae,Mean Qwk,Tr Qwk,Cc Qwk,Lr Qwk,Gra Qwk,Within 0.5 Acc
1,0.052575,0.008973,0.975816,0.250201,0.260744,0.241137,0.260145,0.238779,0.430169
2,0.040522,0.008183,0.918229,0.393318,0.376639,0.396335,0.410449,0.389849,0.474909
3,0.034417,0.007502,0.868501,0.465291,0.474862,0.456742,0.482083,0.447478,0.513906
4,0.028851,0.012118,1.109885,0.360921,0.379410,0.371900,0.367868,0.324507,0.412334
5,0.019051,0.007870,0.860036,0.512362,0.527520,0.513940,0.523579,0.484408,0.526300
6,0.010346,0.008186,0.869710,0.522301,0.530190,0.525362,0.544852,0.488799,0.537183
7,0.007055,0.007814,0.871524,0.507177,0.511534,0.501056,0.525411,0.490708,0.512696
8,0.005946,0.008356,0.888452,0.512398,0.516719,0.517650,0.526060,0.489164,0.519045
9,0.004514,0.008051,0.864722,0.524899,0.532596,0.537173,0.536185,0.493641,0.527509
10,0.004399,0.007472,0.843410,0.507326,0.523074,0.503005,0.522754,0.480473,0.538089


TrainOutput(global_step=2070, training_loss=0.02392350226136797, metrics={'train_runtime': 3840.24, 'train_samples_per_second': 17.233, 'train_steps_per_second': 0.539, 'total_flos': 0.0, 'train_loss': 0.02392350226136797, 'epoch': 10.0})

In [15]:
val_metrics = trainer.evaluate(tokenized_ds["validation"])
print("Validation metrics:", val_metrics)

if "test" in tokenized_ds:
    test_metrics = trainer.evaluate(tokenized_ds["test"], metric_key_prefix="test")
    print("Test metrics:", test_metrics)

Validation metrics: {'eval_loss': 0.008051377721130848, 'eval_mean_mae': 0.8647218942642212, 'eval_mean_qwk': 0.5248986658382726, 'eval_tr_qwk': 0.5325958575605556, 'eval_cc_qwk': 0.5371727118770748, 'eval_lr_qwk': 0.5361847926653852, 'eval_gra_qwk': 0.4936413012500751, 'eval_within_0.5_acc': 0.527509068923821, 'eval_runtime': 34.7268, 'eval_samples_per_second': 23.814, 'eval_steps_per_second': 23.814, 'epoch': 10.0}


early stopping required metric_for_best_model, but did not find eval_mean_qwk so early stopping is disabled


Test metrics: {'test_loss': 0.007501350715756416, 'test_mean_mae': 0.8475241512060165, 'test_mean_qwk': 0.564334881235012, 'test_tr_qwk': 0.5744720507557197, 'test_cc_qwk': 0.5767769816899806, 'test_lr_qwk': 0.5726078394797658, 'test_gra_qwk': 0.5334826530145815, 'test_within_0.5_acc': 0.5265700483091788, 'test_runtime': 35.3558, 'test_samples_per_second': 23.419, 'test_steps_per_second': 23.419, 'epoch': 10.0}


In [ ]:
trainer.save_model(f"{OUTPUT_DIR}/best_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/best_model")